# TP S42 : Mon premier pipeline ETL

**BUT INFO3 Développeur IA, R5.C.06, DATA5, Semaine 42**
M. Sakkari, IUT de La Rochelle

> **Ce TP n'est pas noté.** Durée : 3 h. Gardez votre notebook complété.

## Dans ce TP, on apprend à...
- lire des fichiers CSV et JSON ;
- mesurer la qualité des données ;
- nettoyer, ou rejeter avec un motif ;
- charger dans SQLite sans doublons ;
- vérifier le résultat ;
- relancer le pipeline automatiquement.

> **Comment travailler :**
> - Une cellule « Démo » est déjà écrite : lancez-la et lisez-la.
> - Une cellule « À vous » est à compléter : remplacez les `...` par votre code.
> - Comparez votre résultat avec le « Résultat attendu » écrit dans la cellule.

## Le cas
Le Café Rochelais vend du café, du thé et des accessoires en ligne.
Ses données viennent de trois sources :

| Fichier | Origine |
|---|---|
| `data/clients.csv` | export d'un tableur |
| `data/produits.json` | export du site web |
| `data/commandes.csv` | export de la caisse en ligne |

## Objectif
Construire une base SQLite propre : `cafe.db`.

Les fichiers contiennent des erreurs volontaires.
À vous de les trouver.

Exécutez les cellules dans l'ordre.

## Petit lexique avant de commencer

| Mot | Ce que ça veut dire | Exemple |
|---|---|---|
| **ETL** | Extract, Transform, Load : prélever des données dans plusieurs sources, les nettoyer et les remettre en forme, puis les charger dans une base cible | 3 fichiers en entrée, 1 base propre en sortie |
| **Profilage** | examiner les données et les mesurer (statistiques) pour connaître leur qualité, avant de les corriger | 13 clients sans ville |
| **Rejet** | une ligne qui échoue aux règles de qualité. On ne la charge pas : on la garde dans une table à part (`rejet`), avec la raison et la date | une commande de −3 cafés |
| **Upsert** | combinaison de *update* et *insert* : on met à jour la ligne si elle existe déjà, on la crée sinon. C'est le moyen le plus courant d'obtenir l'idempotence dans un pipeline ETL | une commande déjà en base est mise à jour, pas copiée |
| **Idempotent** | propriété d'un traitement qui produit exactement le même résultat, qu'on l'exécute une seule fois ou plusieurs fois avec les mêmes données d'entrée | après 2 lancements, toujours 1879 commandes |
| **Empreinte** (*hash*) | une courte suite de caractères calculée à partir du contenu d'un fichier. Si le fichier change, l'empreinte change | on ajoute un espace au fichier : l'empreinte change |

**Définitions détaillées :** support de cours S42, diapositives « Vocabulaire » (5 à 10).

In [ ]:
# Outils utilisés dans tout le TP : fichiers, texte, base SQLite, dates et tableaux (pandas)
import json, os, re, sqlite3
from datetime import datetime
import pandas as pd


def brut(ligne):
    """Ligne d'origine (texte) mise en JSON, pour la table rejet."""
    return json.dumps(ligne.to_dict(), ensure_ascii=False)

## Partie A : Extraire (E)
Voir le cours : « Vocabulaire (1/6) », ETL.

### À vous : A.1 lire les fichiers CSV
**À faire :** lisez `clients.csv` et `commandes.csv`.
- Tout doit être lu en texte.
- Une case vide doit rester un texte vide.

Indices : les options `dtype` et `keep_default_na` de `pd.read_csv`.

In [ ]:
# Objectif : lire les deux fichiers CSV, en texte.
# Résultat attendu : deux tableaux, clients (308 lignes) et commandes (2035 lignes).

# Étape 1 : lire data/clients.csv, en texte, sans transformer les cases vides
clients = ...

# Étape 2 : lire data/commandes.csv de la même façon
commandes = ...

### À vous : A.2 lire le fichier JSON
**À faire :** lisez `produits.json` et rangez-le dans un tableau `produits`.

Indices : `json.load` donne une liste de dictionnaires. `pd.DataFrame` sait en faire un tableau.

In [ ]:
# Objectif : lire le catalogue des produits (JSON).
# Résultat attendu : un tableau produits (24 lignes).

with open("data/produits.json", encoding="utf8") as f:
    # Étape 1 : construire un tableau à partir du contenu du fichier JSON
    produits = ...

**A.3 Regarder les données.**
Exécutez la cellule. Elle affiche :
- les 5 premières lignes de chaque tableau ;
- le nombre de lignes et de colonnes de chaque tableau (`shape`).

In [ ]:
# Les 5 premières lignes de chaque tableau
display(clients.head(), commandes.head(), produits.head())

# La taille de chaque tableau : (nombre de lignes, nombre de colonnes)
print("Le tableau clients a", clients.shape[0], "lignes et", clients.shape[1], "colonnes.")
print("Le tableau commandes a", commandes.shape[0], "lignes et", commandes.shape[1], "colonnes.")
print("Le tableau produits a", produits.shape[0], "lignes et", produits.shape[1], "colonnes.")

## Partie B : Profiler
Objectif : mesurer les problèmes avant de les corriger.
Voir le cours : « Vocabulaire (2/6) », profilage.

### À vous : B.1 profiler les clients
**À faire :** répondez à ces six questions sur le tableau `clients`.
Rangez chaque réponse dans la variable indiquée, dans la cellule de code.
1. Combien de clients n'ont pas de ville (la case `ville` est vide) ? (`n_villes_vides`)
2. Combien de villes différentes y a-t-il, avant tout nettoyage ? (`n_villes_brutes`)
3. Combien y en aurait-il après avoir enlevé les espaces en trop et uniformisé la casse ? (`n_villes_nettes`)
4. Combien d'emails sont invalides (sans `@`, ou sans domaine) ? (`n_emails_invalides`) L'expression `EMAIL_RE` est donnée.
5. Combien d'emails sont en double, en ignorant la casse ? (`n_emails_doublons`)
6. Combien de dates sont écrites `JJ/MM/AAAA` (avec des `/`) ? (`n_dates_slash`)

Indices : `.str.strip()`, `.str.title()`, `.nunique()`, `.str.match()`, `.duplicated()`, `.str.contains()`, `.sum()`.

In [ ]:
# Objectif : mesurer les problèmes du fichier clients.
# Résultat attendu : 13, 23, 8, 17, 8 et 40 (dans l'ordre des questions).

EMAIL_RE = re.compile(r"^[^@\s]+@[^@\s]+\.[a-z]{2,}$")   # un texte, un @, un texte, un point, une extension

# Question 1
n_villes_vides = ...

# Questions 2 et 3
n_villes_brutes = ...
n_villes_nettes = ...

# Question 4
n_emails_invalides = ...

# Question 5
n_emails_doublons = ...

# Question 6
n_dates_slash = ...

# Contrôle : vos six réponses
print("Il y a", n_villes_vides, "clients sans ville.")
print("Il y a", n_villes_brutes, "villes différentes avant nettoyage.")
print("Il y a", n_villes_nettes, "villes différentes après nettoyage.")
print("Il y a", n_emails_invalides, "emails invalides.")
print("Il y a", n_emails_doublons, "emails en double (casse ignorée).")
print("Il y a", n_dates_slash, "dates écrites JJ/MM/AAAA.")

**Lecture :** 13 clients n'ont pas de ville. Les 23 écritures de villes se ramènent à 8 villes. 17 emails sont invalides, 8 sont en double, et 40 dates sont écrites avec des `/`.

### À vous : B.2 profiler les commandes
**À faire :** répondez à ces quatre questions sur le tableau `commandes`.
Rangez chaque réponse dans la variable indiquée, dans la cellule de code.
1. Combien de lignes sont entièrement dupliquées ? (`n_lignes_dup`)
2. Combien d'`id_commande` différents y a-t-il ? (`n_ids`)
3. Quelles sont les valeurs différentes de `statut` ? Donnez la liste triée. (`statuts`)
4. Combien de quantités sont négatives ou nulles ? (`n_qte_neg`)

Indices : `.duplicated()`, `.nunique()`, `.unique()`, `sorted(...)`. La colonne `quantite` est du texte : convertissez-la d'abord avec `.astype(int)`.

In [ ]:
# Objectif : mesurer les problèmes du fichier commandes.
# Résultat attendu : 35, 2000, une liste de 7 valeurs et 15 (dans l'ordre des questions).

# Question 1
n_lignes_dup = ...

# Question 2
n_ids = ...

# Question 3
statuts = ...

# Question 4
n_qte_neg = ...

print("Les valeurs différentes de statut sont :", statuts)
print("Il y a", n_lignes_dup, "lignes entièrement dupliquées.")
print("Il y a", n_ids, "commandes différentes (id_commande).")
print("Il y a", n_qte_neg, "quantités négatives ou nulles.")

**Lecture :** 35 lignes sont des doublons exacts, donc il y a 2000 commandes différentes. La colonne `statut` a 7 écritures pour seulement 3 statuts réels. 15 quantités sont négatives ou nulles.

### À vous : B.3 le tableau des problèmes
**À faire :** rassemblez vos résultats de B.1 et B.2 dans un tableau `problemes`.
- Une ligne par problème, dans l'ordre donné dans la cellule.
- Trois colonnes : `Problème`, `Nombre` et `Décision`.
- La colonne `Nombre` vient de vos variables. Ne tapez aucun nombre à la main.
- La colonne `Décision` reste vide pour l'instant. Vous la remplirez en partie C.

Attention à la ligne « Valeurs de statut différentes » : `statuts` est une liste, pas un nombre.
Le nombre à mettre dans le tableau est la taille de cette liste, c'est-à-dire le nombre de valeurs qu'elle contient.

In [ ]:
# Objectif : rassembler vos résultats de B.1 et B.2 dans un seul tableau.
# Résultat attendu : un tableau de 8 lignes et 3 colonnes. La colonne Décision est vide pour l'instant.

problemes = pd.DataFrame({
    "Problème": ["Clients sans ville", "Villes différentes avant nettoyage", "Emails invalides", "Emails en double",
                 "Dates JJ/MM/AAAA", "Lignes de commandes dupliquées", "Valeurs de statut différentes",
                 "Quantités négatives ou nulles"],
    # Étape 1 : les nombres, pris dans vos variables, dans le même ordre que les problèmes
    "Nombre": ...,
    "Décision": "",
})
problemes

## Partie C : Transformer (T)
### Démo : C.1 nettoyer les clients
Cette cellule est donnée. Lisez-la, puis exécutez-la.

**Piège des dates.** `03/04/2026` peut être :
- le 3 avril (JJ/MM/AAAA) ;
- le 4 mars (MM/JJ/AAAA).

On choisit le format selon la source :
- clients : `JJ/MM/AAAA` ;
- commandes : `MM/JJ/AAAA`.

In [ ]:
# Cette cellule nettoie le tableau des clients. Il n'y a rien à compléter.

# Fonction qui lit une date écrite dans l'un des formats donnés et la renvoie au format AAAA-MM-JJ.
def parse_date(s, formats):
    for f in formats:
        try:
            return datetime.strptime(s.strip(), f).date().isoformat()
        except ValueError:
            pass             # ce format ne convient pas : on essaie le suivant
    return None              # aucun format ne convient


clients_nettoyes = clients.copy()           # on travaille sur une copie : le tableau d'origine ne change pas

# Noms en majuscules, prénoms avec une majuscule au début, emails en minuscules
clients_nettoyes["nom"]     = clients_nettoyes.nom.str.strip().str.upper()
clients_nettoyes["prenom"]  = clients_nettoyes.prenom.str.strip().str.title()
clients_nettoyes["email"]   = clients_nettoyes.email.str.strip().str.lower()

# Villes : on enlève les espaces en trop et on uniformise la casse (« NIORT » devient « Niort »).
# Un client sans ville (la case est vide, 13 clients dans le tableau des problèmes) reçoit « Inconnue ».
clients_nettoyes["ville"]   = clients_nettoyes.ville.str.strip().str.title().replace("", "Inconnue")

# Dates d'inscription : le fichier mélange deux formats (AAAA-MM-JJ et JJ/MM/AAAA).
# On les met toutes au format AAAA-MM-JJ.
clients_nettoyes["date_inscription"] = clients_nettoyes.date_inscription.map(lambda s: parse_date(s, ["%Y-%m-%d", "%d/%m/%Y"]))

clients_nettoyes["id_client"] = clients_nettoyes.id_client.astype(int)    # l'identifiant devient un nombre

# Contrôle : nombre de villes différentes, et nombre de dates que la fonction n'a pas su lire
print("Il y a", clients_nettoyes.ville.nunique(), "villes différentes après nettoyage.")
print("Il y a", clients_nettoyes.date_inscription.isna().sum(), "clients dont la date d'inscription est impossible à lire.")

### À vous : C.1.1 les emails invalides
**À faire :** remplacez chaque email invalide par `None`.
Un email est valide s'il passe `EMAIL_RE`.
Le client est conservé : seul son email est vidé.

Indice : la méthode `.where()`.

In [ ]:
# Objectif : vider les emails invalides, sans supprimer le client.
# Résultat attendu : 17 emails vides (None) dans clients_nettoyes.

# Étape 1 : garder l'email s'il passe EMAIL_RE, sinon None
clients_nettoyes["email"] = ...

# Contrôle : nombre d'emails vidés
print("Il y a", clients_nettoyes.email.isna().sum(), "emails invalides remplacés par None.")

**Lecture :** les 17 emails invalides sont remplacés par `None`. Les 17 clients restent dans le tableau.

### À vous : C.1.2 supprimer les clients en double
Un même client apparaît parfois deux fois, avec deux `id_client` différents et le même email.
On garde la ligne qui a le plus petit `id_client`. On supprime l'autre.

**À faire :**
1. Modifiez le dictionnaire `remap`. Il contient des paires `clé : valeur`.
   La clé est l'`id_client` d'un client. La valeur est l'`id_client` du client qu'on garde à sa place.
   Pour chaque client supprimé, remplacez la valeur de sa clé par l'`id_client` du client conservé.
   Exemple : le client 301 est un doublon du client 213.
   - avant : `{213: 213, 301: 301, ...}`
   - après : `{213: 213, 301: 213, ...}`

   La clé 301 ne change pas. Sa valeur passe de 301 à 213.
   Plus loin, `remap` servira à rattacher les commandes du client 301 au client 213.
2. La liste `rejets_clients` dans la cellule de code vide, ci-dessous, sert à garder la trace des clients supprimés.
   Pour chaque client supprimé, ajoutez-y un tuple de 4 éléments :
   - `"clients"` ;
   - l'`id_client` supprimé ;
   - le texte `"doublon_de_"` suivi de l'`id_client` conservé ;
   - la ligne d'origine en JSON : `brut(clients.loc[i])`.
   Exemple pour le client 301, doublon du client 213 :
   ```
   ("clients", 301, "doublon_de_213", '{"id_client": "301", "nom": "Simon", "prenom": "Sarah", "email": "SARAH.SIMON213@EXAMPLE.ORG", "ville": "Saintes", "date_inscription": "2023-01-30"}')
   ```
   Il y a un tuple par client supprimé, et chaque tuple commence par `"clients"` (la table d'origine).
   Avec deux clients supprimés (301, doublon du 213, et 303, doublon du 103), la liste contient deux tuples :
   ```
   rejets_clients = [
       ("clients", 301, "doublon_de_213", '{...}'),
       ("clients", 303, "doublon_de_103", '{...}'),
   ]
   ```
3. Rangez dans `clients_ok` les clients conservés.
   `clients_ok` est un tableau, comme `clients_nettoyes`, qui ne garde que les clients conservés (301 lignes).

Indices :
- comparez seulement les clients qui ont un email ;
- `.duplicated()` repère les lignes en double, et son argument `keep` choisit celle qu'on garde ;
- `.iterrows()` parcourt les lignes : `i` est l'indice de la ligne, `r` est la ligne.

In [ ]:
# Objectif : supprimer les doublons de clients, sans perdre leurs commandes.
# Résultat attendu : 301 clients conservés et 7 clients supprimés.

clients_nettoyes = clients_nettoyes.sort_values("id_client")
remap = {i: i for i in clients_nettoyes.id_client}   # au départ, chaque client est conservé
rejets_clients = []

# Étape 1 : repérer les doublons d'email, puis remplir remap et rejets_clients
...

# Étape 2 : ne garder que les clients conservés
clients_ok = ...

# Contrôle : clients conservés, clients supprimés, et l'exemple du client 301
print("Il y a", len(clients_ok), "clients conservés.")
print("Il y a", len(rejets_clients), "clients supprimés.")
print(f"Le client 301 est remplacé par le client {remap[301]}.")

**Lecture :** il y avait 308 clients. 7 sont supprimés, 301 sont conservés. `remap[301]` vaut 213 : les commandes du client 301 iront au client 213.
- Le profilage (B.1) comptait 8 emails en double, mais seulement 7 clients sont supprimés.
- La raison : les clients 90 et 302 ont un email invalide (`jade.vincent90example.org`, sans `@`). Cet email est vidé en C.1.1, donc on ne peut plus comparer ces deux clients. Les deux restent dans le tableau.

**Question :** pourquoi garder `remap` ?
Pourquoi ne pas faire seulement `drop_duplicates("email")` ?

*Votre réponse :* 

### C.2 Nettoyer les commandes
### À vous : étape 1, les doublons exacts
**À faire :**
- supprimez les lignes entièrement identiques ;
- gardez le nombre de lignes supprimées dans `nb_dup`.

Indice : la méthode `.drop_duplicates()`.

In [ ]:
# Objectif : supprimer les commandes en double exact.
# Résultat attendu : nb_dup = 35, et 2000 lignes restantes.

# Étape 1 : supprimer les lignes entièrement identiques
commandes_sans_doublon = ...

# Étape 2 : combien de lignes avez-vous supprimées ?
nb_dup = ...

# Contrôle : lignes supprimées et lignes restantes
print("On a supprimé", nb_dup, "lignes en double.")
print("Il reste", len(commandes_sans_doublon), "lignes.")

**Lecture :** 35 lignes étaient des doublons exacts. Il reste 2000 commandes.

### Démo : étape 2, convertir et normaliser
Cellule donnée. Exécutez-la.

Regardez le statut de `en cours ` et de `?`. Il devient `NaN`. C'est voulu.

In [ ]:
# Cette cellule prépare les commandes. Il n'y a rien à compléter.

# Les statuts reconnus : les variantes (« Livrée ») sont ramenées à une seule écriture (« livree »).
STATUTS = {"livree": "livree", "livrée": "livree", "expediee": "expediee", "annulee": "annulee"}

commandes_nettoyees = commandes_sans_doublon.copy()      # les commandes, sans les doublons exacts

commandes_nettoyees["quantite"]    = commandes_nettoyees.quantite.astype(int)         # la quantité devient un nombre
commandes_nettoyees["montant_ttc"] = commandes_nettoyees.montant_ttc.astype(float)    # le montant devient un nombre décimal

# Statut : espaces enlevés, minuscules, puis écriture unique. Un statut inconnu (« en cours », « ? ») devient NaN.
commandes_nettoyees["statut"]      = commandes_nettoyees.statut.str.strip().str.lower().map(STATUTS)

# Dates des commandes : deux formats possibles, AAAA-MM-JJ et MM/JJ/AAAA. Une date impossible devient None.
commandes_nettoyees["date_commande"] = commandes_nettoyees.date_commande.map(lambda s: parse_date(s, ["%Y-%m-%d", "%m/%d/%Y"]))

# Un client supprimé comme doublon est remplacé par le client conservé (dictionnaire remap).
commandes_nettoyees["id_client"]   = commandes_nettoyees.id_client.astype(int).map(lambda i: remap.get(i, i))

# Contrôle : nombre de commandes dont le statut est inconnu, et dont la date est impossible à lire
print("Il y a", commandes_nettoyees.statut.isna().sum(), "commandes dont le statut est inconnu (ni livree, ni expediee, ni annulee).")
print("Il y a", commandes_nettoyees.date_commande.isna().sum(), "commandes dont la date est impossible à lire.")

### À vous : étape 3, les règles de rejet
- Une commande est rejetée si elle viole une règle.
- Les règles sont testées dans cet ordre.
- Une ligne qui viole plusieurs règles reçoit un seul motif : celui de la première règle testée.

| # | Motif | Test |
|---|---|---|
| 1 | `date_invalide` | la date vaut `None` |
| 2 | `statut_inconnu` | le statut vaut `NaN` |
| 3 | `quantite_invalide` | `quantite <= 0` |
| 4 | `client_inconnu` | `id_client` absent des clients conservés (`clients_ok`) |
| 5 | `produit_inconnu` | `sku` absent du catalogue |
| 6 | `montant_incoherent` | \|montant_ttc − prix_ht × quantite × (1 + tva/100)\| > 0,015 |

**À faire :** complétez `rejeter(masque, motif)`.
- Pour chaque ligne du masque encore valide, ajoutez `("commandes", id_commande, motif, ligne_brute)` à `rejets_cmd`.
- `ligne_brute` vaut `brut(commandes_sans_doublon.loc[i])` : la ligne d'origine, en JSON.
- Puis retirez ces lignes.
- Appelez-la dans l'ordre du tableau.

**Arrondis.** Le seuil est 0,015 et non 0,01. Un calcul en décimaux peut s'écarter d'un centime sans que la commande soit fausse.

In [ ]:
# Objectif : rejeter les commandes invalides, avec un motif.
# Résultat attendu : 121 rejets (39 + 29 + 19 + 14 + 10 + 10) et 1879 commandes valides.

prix_ht = produits.set_index("sku").prix_ht.astype(float)
tva = produits.set_index("sku").tva.astype(float)
attendu = (commandes_nettoyees.sku.map(prix_ht) * commandes_nettoyees.quantite * (1 + commandes_nettoyees.sku.map(tva) / 100)).round(2)   # montant TTC attendu

rejets_cmd = []
ok = pd.Series(True, index=commandes_nettoyees.index)       # True = la ligne est encore valide


def rejeter(masque, motif):
    global ok
    # Étape 1 : pour chaque ligne rejetée par cette règle, ajouter un tuple à rejets_cmd (format dans la consigne)
    # Étape 2 : retirer ces lignes de ok
    ...


rejeter(commandes_nettoyees.date_commande.isna(), "date_invalide")

# Étape 3 : appeler rejeter pour les 5 autres règles du tableau, dans l'ordre
...

commandes_ok = commandes_nettoyees[ok]

# Contrôle : le nombre de commandes rejetées pour chaque motif, puis le nombre de commandes valides
print("Il y a", len(rejets_cmd), "commandes rejetées. Nombre par motif :")
print(pd.Series([motif for _, _, motif, _ in rejets_cmd]).value_counts().to_string())
print("Il y a", len(commandes_ok), "commandes valides.")

**Lecture :** le tableau donne le nombre de commandes rejetées pour chaque motif.
- Il y a 121 commandes rejetées et 1879 commandes valides.
- Une commande n'a qu'un motif : celui de la première règle qu'elle viole.

### À vous : C.2 bis les décisions
Pour chaque problème du tableau `problemes`, vous avez maintenant une décision à prendre.

**Corriger ou rejeter ?**
- On corrige si la bonne valeur est évidente : « LA ROCHELLE » devient « La Rochelle ».
- On rejette si on ne peut pas connaître la bonne valeur : quantité −3, produit inexistant.
- Une ligne rejetée est gardée dans la table `rejet`, avec son motif, sa date et son contenu d'origine.
- Voir le cours : « Vocabulaire (3/6) », rejet.

**À faire :** remplissez la colonne `Décision` du tableau `problemes`, une décision par problème.
- Utilisez les verbes « corriger », « dédoublonner » ou « rejeter ».
- Ajoutez en quelques mots comment, par exemple « corriger : un seul format ».
- Gardez l'ordre des lignes du tableau.

In [ ]:
# Objectif : noter la décision prise pour chaque problème du tableau.
# Résultat attendu : le tableau problemes, avec la colonne Décision remplie (8 lignes).

# Étape 1 : une décision par problème, dans le même ordre que le tableau
problemes["Décision"] = ...

problemes

### À vous : C.3 noter tous les motifs
Une ligne peut violer plusieurs règles.
La table `rejet` ne garde que le premier motif.

**À faire :**
- construisez un tableau `motifs` : une colonne par règle, `True` si la ligne viole la règle ;
- comptez les règles violées par chaque ligne ;
- comptez les lignes qui en violent au moins deux ;
- pour ces lignes, écrivez la liste de tous les motifs.

Indice : la méthode `.sum()` et son argument `axis`.

In [ ]:
# Objectif : repérer les lignes qui violent plusieurs règles.
# Résultat attendu : 18 lignes violent au moins 2 règles ; 19 motifs ne sont pas notés dans la table rejet.

# Étape 1 : un tableau de 6 colonnes (une par règle du tableau de l'étape 3), True si la ligne viole la règle
motifs = pd.DataFrame({
    "date_invalide": ...,
    "statut_inconnu": ...,
    "quantite_invalide": ...,
    "client_inconnu": ...,
    "produit_inconnu": ...,
    "montant_incoherent": ...,
})

# Étape 2 : le nombre de règles violées par chaque ligne
nb_regles = ...

# Étape 3 : le nombre de lignes qui violent au moins 2 règles
n_multi = ...

# Étape 4 : le nombre de motifs non notés (règles violées en plus de la première)
n_perdus = ...

# Étape 5 : pour les lignes qui violent au moins 2 règles, tous les motifs dans une chaîne "motif1, motif2"
tous_les_motifs = ...

print("Il y a", n_multi, "lignes qui violent au moins 2 règles.")
print("Il y a", n_perdus, "motifs qui ne sont pas notés dans la table rejet.")
print("Pour ces lignes, voici tous les motifs :")
print(tous_les_motifs)

**Question :** que gagne-t-on à noter tous les motifs ?

*Votre réponse :* 

## Partie D : Charger (L)
### Démo : D.1 créer le schéma
Clés et contraintes protègent la base contre vos erreurs.
Cellule donnée. Lisez-la, puis exécutez-la.

In [ ]:
# Cette cellule crée la base. Il n'y a rien à compléter.

# Les 5 tables de la base. Une clé primaire (PRIMARY KEY) identifie une ligne.
# Une clé étrangère (REFERENCES) renvoie à une ligne d'une autre table.
# CHECK et UNIQUE refusent les valeurs interdites.
SCHEMA = """
CREATE TABLE IF NOT EXISTS client (id_client INTEGER PRIMARY KEY, nom TEXT, prenom TEXT, email TEXT UNIQUE, ville TEXT, date_inscription TEXT);
CREATE TABLE IF NOT EXISTS produit (sku TEXT PRIMARY KEY, libelle TEXT, categorie TEXT, prix_ht REAL, tva REAL);
CREATE TABLE IF NOT EXISTS commande (id_commande INTEGER PRIMARY KEY, id_client INTEGER NOT NULL REFERENCES client(id_client),
  sku TEXT NOT NULL REFERENCES produit(sku), quantite INTEGER CHECK(quantite > 0), date_commande TEXT, montant_ttc REAL, statut TEXT);
CREATE TABLE IF NOT EXISTS rejet (source TEXT, cle INTEGER, motif TEXT, execution TEXT, ligne_brute TEXT, date_rejet TEXT);
CREATE TABLE IF NOT EXISTS etl_log (execution TEXT, fichier TEXT, lus INTEGER, charges INTEGER, rejetes INTEGER, doublons INTEGER, declencheur TEXT);
"""

try:
    con.close()              # si on relance la cellule
except NameError:
    pass
if os.path.exists("cafe.db"):
    os.remove("cafe.db")     # on repart d'une base vide

con = sqlite3.connect("cafe.db")          # on ouvre (et on crée) la base cafe.db
con.executescript(SCHEMA)                 # on crée les tables
con.execute("PRAGMA foreign_keys = ON")   # SQLite vérifie maintenant les clés étrangères

**Question :** citez deux erreurs de la partie C que le schéma aurait bloquées.

*Votre réponse :* 

### Démo : D.2 charger avec un upsert
- La ligne n'existe pas : elle est insérée.
- La clé existe déjà : la ligne est mise à jour.
- Syntaxe : `ON CONFLICT ... DO UPDATE`.
- Voir le cours : « Vocabulaire (4/6) », upsert.

Voici le chargement des clients (donné) :

In [ ]:
# Cette cellule charge les clients. Il n'y a rien à compléter.

# INSERT ajoute un client. Si son id_client existe déjà (ON CONFLICT), ses colonnes sont mises à jour.
con.executemany("""INSERT INTO client VALUES (?,?,?,?,?,?)
  ON CONFLICT(id_client) DO UPDATE SET nom=excluded.nom, prenom=excluded.prenom,
    email=excluded.email, ville=excluded.ville, date_inscription=excluded.date_inscription""",
  clients_ok[["id_client", "nom", "prenom", "email", "ville", "date_inscription"]].itertuples(index=False, name=None))

con.commit()      # on enregistre définitivement dans la base

### À vous : D.2 charger les autres tables
**À faire :** chargez dans cet ordre.
1. `produit` : `produits`, upsert sur `sku`.
2. `commande` : `commandes_ok`, upsert sur `id_commande`.
3. `rejet` : `rejets_clients + rejets_cmd`, simple `INSERT`, avec 6 colonnes :
   - `source`, `cle`, `motif` (du tuple) ;
   - `execution` : l'étiquette `"run1"` ;
   - `ligne_brute` : la ligne d'origine en JSON (du tuple) ;
   - `date_rejet` : `datetime.now().isoformat(timespec="seconds")`.

**Pourquoi cet ordre ?**
- `commande` référence `client` et `produit`.
- On charge toujours les tables parents avant les tables enfants.
- Essayez `commande` en premier : vous verrez l'erreur.

In [ ]:
# Objectif : charger les trois autres tables.
# Résultat attendu : produit 24 lignes, commande 1879 lignes, rejet 128 lignes.

# Étape 1 : table produit (upsert sur sku)
...

# Étape 2 : table commande (upsert sur id_commande)
...

# Étape 3 : table rejet (simple INSERT, 6 colonnes, avec "run1", la ligne brute et la date)
...

con.commit()

# Contrôle : nombre de lignes dans chaque table
for table in ["client", "produit", "commande", "rejet"]:
    print("La table", table, "contient", con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0], "lignes.")

**Lecture :** la base contient 301 clients, 24 produits et 1879 commandes. La table `rejet` a 128 lignes : 121 commandes rejetées et 7 clients supprimés.

## Partie E : Vérifier
Objectif : prouver que rien n'a été perdu ni inventé.
Voir le cours : « Vérifier » (diapositive 14).

### À vous : E.1 le bilan
**À faire :** calculez lues − doublons exacts − rejetées − chargées.
Le résultat doit être 0.

In [ ]:
# Objectif : vérifier que le bilan tombe juste.
# Résultat attendu : 2035 - 35 - 121 = 1879, écart 0.

# Étape 1 : nombre de commandes lues, rejetées et chargées
lues = ...
rejetees = ...
chargees = ...

# Étape 2 : l'écart doit valoir 0
ecart = ...

print("On a lu", lues, "lignes, supprimé", nb_dup, "doublons exacts, rejeté", rejetees, "commandes et chargé", chargees, "commandes.")
print("L'écart vaut", ecart, "(il doit valoir 0).")

### À vous : E.2 contrôles SQL
**À faire :** écrivez trois requêtes.
- commandes sans client (doit donner 0) ;
- chiffre d'affaires TTC ;
- nombre de commandes par statut.

In [ ]:
# Objectif : contrôler la base avec trois requêtes SQL.
# Résultat attendu : 0 orphelin, chiffre d'affaires 83575.52, trois statuts.

# Étape 1 : le nombre de commandes dont le client n'existe pas (indice : une jointure LEFT JOIN)
orphelins = ...

# Étape 2 : le chiffre d'affaires total, arrondi à 2 décimales (indice : SUM)
chiffre_affaires = ...

# Étape 3 : le nombre de commandes par statut, dans un dictionnaire (indice : GROUP BY)
statuts_base = ...

print("Il y a", orphelins, "commandes sans client.")
print("Le chiffre d'affaires total est de", chiffre_affaires, "euros TTC.")
print("Nombre de commandes par statut :", statuts_base)

### À vous : E.3 retrouver une ligne rejetée
La commande 588 a été rejetée. Que contenait-elle ?

**À faire :**
- lisez sa ligne dans la table `rejet` (`source = 'commandes'`, `cle = 588`) ;
- lisez le motif, la date de rejet et la ligne brute ;
- transformez la ligne brute en dictionnaire avec `json.loads`.

**Question :** pourquoi cette commande est-elle rejetée ?

In [ ]:
# Objectif : retrouver une commande rejetée, avec sa raison et sa date.
# Résultat attendu : le motif client_inconnu, une date, et un dictionnaire (la ligne d'origine).

# Étape 1 : lire dans la table rejet la ligne de la commande 588 : son motif, sa date et sa ligne brute
ligne = ...

# Étape 2 : isoler le motif et la date, puis transformer la ligne brute en dictionnaire
motif_588 = ...
date_588 = ...
commande_588 = ...

print(f"La commande 588 a été rejetée le {date_588}, motif : {motif_588}.")
print("Sa ligne d'origine :", commande_588)

*Votre réponse :* 

## Partie F : Pour aller plus loin
Voir le cours : « Journaliser » (diapositive 15).

### À vous : F.1 un vrai pipeline
- Jusqu'ici, le code est dans des cellules.
- On le range dans un module : `etl.py`.
- Ainsi on pourra le relancer, même automatiquement (partie G).
- La cellule suivante écrit ce fichier (`%%writefile`).
- Son contenu part dans `etl.py` : il n'est pas exécuté dans le notebook.

**À faire :** complétez 3 fonctions.
- `transform_clients` : recopiez votre code de C.1.
- `transform_commandes` : recopiez votre code de C.2.
- `pipeline` : enchaînez extract, transform, load.

Les autres fonctions sont données.

In [ ]:
%%writefile etl.py
"""Pipeline ETL du Café Rochelais (module importé par le notebook)."""
import json, os, re, sqlite3
from datetime import datetime
import pandas as pd

DATA = "data"
DB = "cafe.db"
EMAIL_RE = re.compile(r"^[^@\s]+@[^@\s]+\.[a-z]{2,}$")
STATUTS = {"livree": "livree", "livrée": "livree", "expediee": "expediee", "annulee": "annulee"}
SCHEMA = """
CREATE TABLE IF NOT EXISTS client (id_client INTEGER PRIMARY KEY, nom TEXT, prenom TEXT, email TEXT UNIQUE, ville TEXT, date_inscription TEXT);
CREATE TABLE IF NOT EXISTS produit (sku TEXT PRIMARY KEY, libelle TEXT, categorie TEXT, prix_ht REAL, tva REAL);
CREATE TABLE IF NOT EXISTS commande (id_commande INTEGER PRIMARY KEY, id_client INTEGER NOT NULL REFERENCES client(id_client),
  sku TEXT NOT NULL REFERENCES produit(sku), quantite INTEGER CHECK(quantite > 0), date_commande TEXT, montant_ttc REAL, statut TEXT);
CREATE TABLE IF NOT EXISTS rejet (source TEXT, cle INTEGER, motif TEXT, execution TEXT, ligne_brute TEXT, date_rejet TEXT);
CREATE TABLE IF NOT EXISTS etl_log (execution TEXT, fichier TEXT, lus INTEGER, charges INTEGER, rejetes INTEGER, doublons INTEGER, declencheur TEXT);
"""

def brut(ligne):
    """Ligne d'origine (texte) mise en JSON, pour la table rejet."""
    return json.dumps(ligne.to_dict(), ensure_ascii=False)


def parse_date(s, fmts):
    for f in fmts:
        try:
            return datetime.strptime(s.strip(), f).date().isoformat()
        except ValueError:
            pass
    return None

def extract(commandes_csv="commandes.csv"):
    clients = pd.read_csv(f"{DATA}/clients.csv", dtype=str, keep_default_na=False)
    commandes = pd.read_csv(f"{DATA}/{commandes_csv}", dtype=str, keep_default_na=False)
    with open(f"{DATA}/produits.json", encoding="utf8") as f:
        produits = pd.DataFrame(json.load(f))
    return clients, commandes, produits

def profilage(clients, commandes):
    print("--- Profilage clients ---")
    print("lignes :", len(clients), "| id distincts :", clients.id_client.nunique())
    print("clients sans ville :", (clients.ville.str.strip() == "").sum())
    print("villes distinctes brutes :", clients.ville.nunique(),
          "| après nettoyage :", clients.ville.str.strip().str.title().nunique())
    print("emails invalides :", (~clients.email.str.lower().str.match(EMAIL_RE)).sum())
    print("emails en doublon (casse ignorée) :", clients.email.str.lower().duplicated().sum())
    print("dates JJ/MM/AAAA :", clients.date_inscription.str.contains("/").sum())
    print("--- Profilage commandes ---")
    print("lignes :", len(commandes), "| id distincts :", commandes.id_commande.nunique())
    print("lignes entièrement dupliquées :", commandes.duplicated().sum())
    print("statuts bruts :", sorted(commandes.statut.unique()))
    print("quantités <= 0 :", (commandes.quantite.astype(int) <= 0).sum())

def load(con, clients_ok, produits, commandes_ok, rejets, execution, fichier, lus, nb_dup, declencheur="manuel"):
    con.executescript(SCHEMA)
    con.execute("PRAGMA foreign_keys = ON")
    con.executemany(
        """INSERT INTO client VALUES (?,?,?,?,?,?) ON CONFLICT(id_client) DO UPDATE SET
           nom=excluded.nom, prenom=excluded.prenom, email=excluded.email,
           ville=excluded.ville, date_inscription=excluded.date_inscription""",
        clients_ok[["id_client", "nom", "prenom", "email", "ville", "date_inscription"]].itertuples(index=False, name=None))
    con.executemany(
        """INSERT INTO produit VALUES (?,?,?,?,?) ON CONFLICT(sku) DO UPDATE SET
           libelle=excluded.libelle, categorie=excluded.categorie, prix_ht=excluded.prix_ht, tva=excluded.tva""",
        produits[["sku", "libelle", "categorie", "prix_ht", "tva"]].itertuples(index=False, name=None))
    con.executemany(
        """INSERT INTO commande VALUES (?,?,?,?,?,?,?) ON CONFLICT(id_commande) DO UPDATE SET
           id_client=excluded.id_client, sku=excluded.sku, quantite=excluded.quantite,
           date_commande=excluded.date_commande, montant_ttc=excluded.montant_ttc, statut=excluded.statut""",
        commandes_ok[["id_commande", "id_client", "sku", "quantite", "date_commande", "montant_ttc", "statut"]].itertuples(index=False, name=None))
    maintenant = datetime.now().isoformat(timespec="seconds")
    con.executemany("INSERT INTO rejet VALUES (?,?,?,?,?,?)",
                    [(s, int(k), m, execution, b, maintenant) for s, k, m, b in rejets])
    rej_cmd = sum(1 for r in rejets if r[0] == "commandes")
    con.execute("INSERT INTO etl_log VALUES (?,?,?,?,?,?,?)",
                (execution, fichier, lus, len(commandes_ok), rej_cmd, nb_dup, declencheur))
    con.commit()


def transform_clients(clients):
    """Reçoit la table brute des clients. Renvoie (clients_ok, remap, rejets_clients)."""
    # Objectif : nettoyer les clients, comme dans la partie C.1.
    # Résultat attendu : un tuple (clients_ok, remap, rejets_clients).

    # Étape 1 : recopiez votre code de C.1 (nettoyage, email invalide -> None, dédoublonnage)
    ...


def transform_commandes(commandes, clients_ok, remap, produits):
    """Renvoie (commandes_ok, rejets_cmd, nb_dup)."""
    # Objectif : nettoyer les commandes, comme dans la partie C.2.
    # Résultat attendu : un tuple (commandes_ok, rejets_cmd, nb_dup).

    # Étape 1 : recopiez votre code de C.2 (doublons exacts, conversions, règles de rejet)
    ...


def pipeline(con, fichier, execution, verbose=False, declencheur="manuel"):
    """Enchaîne E, T, L. Renvoie (lus, charges, rejets, nb_dup)."""
    # Objectif : enchaîner extraire, transformer, charger.
    # Résultat attendu : un tuple (lus, charges, rejets, nb_dup).

    # Étape 1 : extract(fichier) donne clients, commandes, produits
    # Étape 2 : transform_clients, puis transform_commandes
    # Étape 3 : load(con, clients_ok, produits, commandes_ok, rejets, execution, fichier, lus, nb_dup, declencheur)
    ...


In [ ]:
import importlib, etl
importlib.reload(etl)       # charge le module etl.py qui vient d'être écrit (à relancer après chaque modification)

### À vous : F.2 l'idempotence
Un pipeline est idempotent si deux exécutions donnent le même résultat qu'une seule.
Voir le cours : « Vocabulaire (5/6) », idempotence.

**À faire :** relancez le pipeline sur les mêmes fichiers, avec l'étiquette `"run2"`.

In [ ]:
# Objectif : relancer le pipeline sur les mêmes fichiers.
# Résultat attendu : toujours 1879 commandes, et 256 lignes dans rejet.

# Étape 1 : lancer le pipeline sur commandes.csv, avec l'étiquette "run2"
...

# Contrôle : nombre de lignes dans les tables commande et rejet
print("La table commande contient", con.execute("SELECT COUNT(*) FROM commande").fetchone()[0], "lignes.")
print("La table rejet contient", con.execute("SELECT COUNT(*) FROM rejet").fetchone()[0], "lignes.")

**Lecture :** il y a toujours 1879 commandes : le pipeline est idempotent. La table `rejet` a doublé (256 lignes), car elle garde les rejets de chaque exécution.

**Question :** que se serait-il passé avec un simple `INSERT` (sans `ON CONFLICT`) ?

*Votre réponse :* 

### À vous : F.3 le nouveau fichier de commandes
`data/commandes_increment.csv` contient 100 commandes :
- 40 déjà reçues ;
- 60 nouvelles.

**À faire :** lancez le pipeline avec l'étiquette `"run3"`.

In [ ]:
# Objectif : charger le nouveau fichier de commandes.
# Résultat attendu : 1939 commandes (1879 + 60 nouvelles).

# Étape 1 : lancer le pipeline sur commandes_increment.csv, étiquette "run3"
...

# Contrôle : nombre de commandes en base
print("La table commande contient", con.execute("SELECT COUNT(*) FROM commande").fetchone()[0], "lignes.")

**Lecture :** il y a maintenant 1939 commandes, soit 60 de plus : les 60 nouvelles. Les 38 commandes déjà connues et valides ont été mises à jour, pas ajoutées une deuxième fois.

### À vous : F.4 les commandes rejetées
Sur les 40 commandes déjà connues :
- 38 sont mises à jour ;
- 2 sont rejetées.

**À faire :** retrouvez ces 2 commandes dans la table `rejet` (exécution `run3`, source `commandes`).
Mettez leurs identifiants dans `ids_rejetes`.

In [ ]:
# Objectif : retrouver les 2 commandes rejetées au run3.
# Résultat attendu : un ensemble de deux identifiants.

# Étape 1 : lire dans la table rejet les clés des commandes rejetées à l'exécution run3
ids_rejetes = ...

print("Les commandes rejetées sont :", ids_rejetes)

**Question :** pourquoi ces 2 commandes étaient-elles déjà absentes de la base après `run1` ?

*Votre réponse :* 

### À vous : F.5 vérifier le fichier source
Voir le cours : « Vérifier » (diapositive 14).

Avant de lancer le pipeline, on contrôle le fichier.
Si le fichier est abîmé, on s'arrête avec un message clair.

**À faire :** écrivez `verifier_source(chemin, colonnes)`.
Elle renvoie la liste des problèmes. La liste est vide si tout va bien.
- le fichier n'existe pas : `"fichier introuvable"` ;
- le fichier n'a aucune ligne de données : `"fichier vide"` ;
- des colonnes manquent : `"colonnes manquantes : a, b"`.

Indices :
- `os.path.exists` ;
- `pd.read_csv` lève `pd.errors.EmptyDataError` si le fichier est totalement vide ;
- `df.columns` donne les colonnes du tableau.

In [ ]:
# Objectif : contrôler un fichier CSV avant de lancer le pipeline.
# Résultat attendu : une liste vide pour un bon fichier, sinon la liste des problèmes trouvés.

def verifier_source(chemin, colonnes):
    problemes = []
    # Étape 1 : si le fichier n'existe pas, renvoyer ["fichier introuvable"]
    # Étape 2 : lire le fichier ; s'il est totalement vide ou sans ligne, ajouter "fichier vide"
    # Étape 3 : si des colonnes attendues manquent, ajouter "colonnes manquantes : a, b"
    ...
    return problemes

**Démo : tester `verifier_source`.**
La cellule suivante crée deux fichiers abîmés, puis les contrôle.
Exécutez-la et lisez les quatre résultats.

In [ ]:
# Cette cellule teste verifier_source. Il n'y a rien à compléter.

# Les colonnes que doit avoir un fichier de commandes
COLONNES_COMMANDES = ["id_commande", "id_client", "sku", "quantite", "date_commande", "montant_ttc", "statut"]

# On crée deux fichiers abîmés pour le test, dans le dossier data_abime
os.makedirs("data_abime", exist_ok=True)
with open("data_abime/vide.csv", "w", encoding="utf8") as f:
    f.write(",".join(COLONNES_COMMANDES) + "\n")                       # l'en-tête seulement : aucune ligne de données
with open("data_abime/sans_montant.csv", "w", encoding="utf8") as f:
    f.write("id_commande,id_client,sku,quantite,date_commande,statut\n1,1,P001,1,2026-01-01,livree\n")   # sans la colonne montant_ttc

# On contrôle un bon fichier, un fichier absent et les deux fichiers abîmés
for chemin in ["data/commandes.csv", "data_abime/inexistant.csv", "data_abime/vide.csv", "data_abime/sans_montant.csv"]:
    print("Problèmes trouvés dans", chemin, ":", verifier_source(chemin, COLONNES_COMMANDES))

### À vous : F.6 le journal d'événements
Voir le cours : « Journaliser » (diapositive 15).

La table `etl_log` résume chaque exécution.
Un journal d'événements raconte ce qui se passe, étape par étape.
Il est écrit dans le fichier `etl.log`.

**À faire :** écrivez `pipeline_surveille(con, fichier, execution)`.
1. Elle vérifie le fichier avec `verifier_source`. S'il y a des problèmes, elle écrit une **erreur** dans le journal et s'arrête.
2. Elle écrit une **information** avant de lancer `etl.pipeline`.
3. Elle écrit une **information** avec le nombre de lignes lues, chargées et de doublons exacts.
4. Elle écrit un **avertissement** par motif de rejet, avec le nombre de lignes.

Indices :
- `journal.info(...)`, `journal.warning(...)`, `journal.error(...)` ;
- `etl.pipeline` renvoie `(lus, charges, rejets, nb_dup)` ;
- pour les motifs, interrogez la table `rejet` avec `GROUP BY` ;
- les motifs `doublon_de_...` se regroupent sous `doublon_client`.

In [ ]:
# Objectif : écrire un journal d'événements dans le fichier etl.log.
# Résultat attendu : dans etl.log, des lignes INFO, des lignes WARNING (une par motif de rejet) et une ligne ERROR pour un fichier abîmé.

import logging

logging.basicConfig(filename="etl.log", encoding="utf8", level=logging.INFO, force=True,
                    format="%(asctime)s %(levelname)s %(message)s")
journal = logging.getLogger("etl")


def pipeline_surveille(con, fichier, execution):
    chemin = "data/" + fichier
    # Étape 1 : verifier_source(chemin, COLONNES_COMMANDES) ; s'il y a des problèmes : journal.error(...) puis return None
    # Étape 2 : journal.info(...) pour annoncer le début, puis lancer etl.pipeline(con, fichier, execution)
    # Étape 3 : journal.info(...) avec les lignes lues, chargées et les doublons exacts
    # Étape 4 : un journal.warning(...) par motif de rejet de cette exécution (table rejet, GROUP BY)
    ...

**Démo : lancer le pipeline surveillé.**
Un premier lancement avec un bon fichier, un second avec un fichier qui n'existe pas.
Lisez ensuite le journal.

In [ ]:
# Cette cellule lance pipeline_surveille. Il n'y a rien à compléter.

# Un bon fichier : le pipeline tourne, et le journal reçoit des informations et des avertissements
pipeline_surveille(con, "commandes_increment.csv", "run4")

# Un fichier absent : le pipeline s'arrête, et le journal reçoit une erreur
pipeline_surveille(con, "inexistant.csv", "run5")

# On affiche le contenu du journal
print("Contenu du journal etl.log :")
print(open("etl.log", encoding="utf8").read())

## Partie G : Automatiser avec Python (20 min)
Objectif : le pipeline se relance seul quand un fichier change.

- La caisse dépose un fichier quand elle veut.
- Un surveillant regarde les fichiers.
- S'ils changent, il appelle `pipeline()`.
- Rien à installer.

### À vous : G.1 l'empreinte
- La date de modification ne suffit pas : réenregistrer un fichier la change, pas son contenu.
- On calcule une empreinte (hash) du contenu.
- Une seule lettre modifiée change l'empreinte.
- Voir le cours : « Vocabulaire (6/6) », empreinte.

**À faire :** écrivez `empreinte(chemin)` avec `hashlib.md5`.

Remarque : MD5 suffit pour détecter un changement. Il ne convient pas pour la sécurité (SHA-256 est recommandé).

In [ ]:
# Objectif : calculer l'empreinte du contenu d'un fichier.
# Résultat attendu : une chaîne de 32 caractères, différente si le contenu change.

import hashlib


def empreinte(chemin):
    # Étape 1 : lire le contenu du fichier, en binaire
    # Étape 2 : calculer son empreinte MD5 (module hashlib) et la renvoyer sous forme de texte
    ...

# Contrôle : l'empreinte d'un fichier, calculée deux fois
print("Première empreinte de data/clients.csv :", empreinte("data/clients.csv"))
print("Seconde empreinte de data/clients.csv :", empreinte("data/clients.csv"))

**Lecture :** le même fichier donne la même empreinte (32 caractères). Si le contenu change, même d'une seule lettre, l'empreinte change.

### À vous : G.2 le surveillant
- Une boucle infinie bloquerait le notebook.
- On écrit donc un script : `surveiller.py`.
- On le lance dans un terminal.
- La cellule suivante écrit ce fichier. Remplacez les `...`.

**À faire :** le surveillant doit
1. mémoriser l'empreinte des 3 fichiers (`clients.csv`, `produits.json`, `commandes_increment.csv`) ;
2. attendre `intervalle` secondes (`time.sleep`), puis recalculer les empreintes ;
3. si l'une a changé, appeler `etl.pipeline("commandes_increment.csv", "auto1", declencheur="modification:" + nom_du_fichier)` ;
4. mémoriser les nouvelles empreintes ;
5. s'arrêter avec `Ctrl+C`.

Le numéro `auto1`, `auto2`... augmente à chaque relance.

In [ ]:
%%writefile surveiller.py
"""Surveille les fichiers sources et relance le pipeline quand le contenu change.
Lancer depuis le dossier du notebook :  python surveiller.py   (Ctrl+C pour arrêter)"""
import hashlib, sqlite3, time
import etl

FICHIERS = ["data/clients.csv", "data/produits.json", "data/commandes_increment.csv"]


def empreinte(chemin):
    # Objectif : calculer l'empreinte (hash) du contenu d'un fichier.
    # Résultat attendu : une chaîne de 32 caractères.

    # Étape 1 : recopiez votre fonction de G.1
    ...


def surveiller(intervalle=2):
    con = sqlite3.connect("cafe.db")
    n = 0
    # Étape 1 : mémoriser l'empreinte de chaque fichier de FICHIERS (un dictionnaire)
    etat = ...
    try:
        while True:
            time.sleep(intervalle)
            # Étape 2 : recalculer les empreintes
            nouvel_etat = ...
            # Étape 3 : si l'une a changé, faire n += 1, puis
            #           etl.pipeline(con, "commandes_increment.csv", f"auto{n}", declencheur="modification:" + nom_du_fichier)
            #           et mémoriser les nouvelles empreintes
            ...
    except KeyboardInterrupt:
        print("Arrêt demandé.")


if __name__ == "__main__":
    surveiller()


### À vous : G.3 tester
Dans un terminal : `python surveiller.py`.

Puis, ailleurs :
- **a)** ajoutez une ligne valide à la fin de `data/commandes_increment.csv` :
  - nouvel `id_commande` ;
  - client 1, produit `P001`, quantité 2 ;
  - date `2026-10-01` ;
  - montant TTC 20,05 ;
  - statut `livree`.
- **b)** rouvrez et enregistrez `data/clients.csv`, sans rien changer.

Résultat attendu :
- après a) : un message dans le terminal, une ligne `auto1` dans `etl_log`, une commande de plus (1940) ;
- après b) : rien ne se passe, car l'empreinte est identique.

Vérifiez dans le notebook :

In [ ]:
# Nouvelle connexion à la base : surveiller.py y a écrit pendant ce temps
con2 = sqlite3.connect("cafe.db")

# Nombre de commandes en base
print("La table commande contient", con2.execute("SELECT COUNT(*) FROM commande").fetchone()[0], "lignes.")

# Une ligne par exécution du pipeline, avec la raison du lancement (colonne declencheur)
print("Exécutions du pipeline (exécution, lues, chargées, déclencheur) :",
      con2.execute("SELECT execution, lus, charges, declencheur FROM etl_log").fetchall())

con2.close()

### À vous : G.4 un piège
Un gros fichier peut arriver pendant que le surveillant le lit.

- Que se passe-t-il s'il lit un fichier à moitié écrit ?
- Proposez une règle simple pour attendre que le fichier ne bouge plus.
- Indice : `os.path.getmtime` et `time.sleep`.

*Votre réponse :* 

**Et la suite ?** Ce surveillant a des limites.
- Il doit rester ouvert.
- Il ne repart pas après une panne.
- Il ne prévient personne si le pipeline échoue.

En S45, nous verrons des outils d'automatisation.

## Bilan
Répondez en une ligne chacune.
1. Quelle erreur vous a le plus surpris ?
2. Une erreur corrigée, une erreur rejetée : pourquoi ?
3. À quoi sert la table `rejet` ?
4. Pourquoi l'idempotence est-elle indispensable ?
5. Que manque-t-il pour la production ?

*Votre réponse :* 